In [1]:
# =========================================================
# 1. MOUNT GOOGLE DRIVE
# =========================================================

from google.colab import drive
drive.mount('/content/drive')

# =========================================================
# 2. IMPORT LIBRARIES
# =========================================================

import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

from tensorflow.keras.applications import MobileNetV3Large

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

from google.colab import files
from PIL import Image

# =========================================================
# 3. SPEED OPTIMIZATION
# =========================================================

tf.keras.mixed_precision.set_global_policy('mixed_float16')

# =========================================================
# 4. DATASET PATHS
# =========================================================

BASE_PATH = "/content/drive/MyDrive/dataset"

TRAIN_PATH = os.path.join(BASE_PATH, "train")
VAL_PATH   = os.path.join(BASE_PATH, "val")
TEST_PATH  = os.path.join(BASE_PATH, "test")

IMG_SIZE = (224, 224)

BATCH_SIZE = 32

EPOCHS_HEAD = 15
EPOCHS_FINE = 15

# =========================================================
# 5. LOAD DATASET
# =========================================================

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    TRAIN_PATH,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    VAL_PATH,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    TEST_PATH,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_ds.class_names

NUM_CLASSES = len(class_names)

print("\nClasses:", class_names)

# =========================================================
# 6. DATASET OPTIMIZATION
# =========================================================

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)

val_ds = val_ds.cache().prefetch(AUTOTUNE)

test_ds = test_ds.prefetch(AUTOTUNE)

# =========================================================
# 7. DATA AUGMENTATION
# =========================================================

data_augmentation = tf.keras.Sequential([

    layers.RandomRotation(0.03),

    layers.RandomZoom(0.05),

    layers.RandomContrast(0.1),

])

# =========================================================
# 8. BUILD MODEL
# =========================================================

base_model = MobileNetV3Large(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

inputs = layers.Input(shape=IMG_SIZE + (3,))

x = data_augmentation(inputs)

x = tf.keras.applications.imagenet_utils.preprocess_input(x)

x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)

x = layers.BatchNormalization()(x)

x = layers.Dense(128, activation="relu")(x)

x = layers.Dropout(0.25)(x)

outputs = layers.Dense(
    NUM_CLASSES,
    activation="softmax",
    dtype="float32"
)(x)

model = models.Model(inputs, outputs)

# =========================================================
# 9. COMPILE MODEL
# =========================================================

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# =========================================================
# 10. CALLBACKS
# =========================================================

callbacks = [

    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),

    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=2,
        verbose=1
    ),

    ModelCheckpoint(
        "MobileNetV3Large_Best.keras",
        save_best_only=True,
        monitor='val_accuracy',
        mode='max'
    )
]

# =========================================================
# 11. TRAIN HEAD
# =========================================================

print("\n==============================")
print("TRAINING HEAD")
print("==============================")

history_head = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD,
    callbacks=callbacks
)

# =========================================================
# 12. FINE TUNING
# =========================================================

print("\n==============================")
print("FINE TUNING")
print("==============================")

base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FINE,
    callbacks=callbacks
)

# =========================================================
# 13. TEST EVALUATION
# =========================================================

print("\n==============================")
print("TESTING MODEL")
print("==============================")

y_true = []
y_pred = []

for images, labels in test_ds:

    predictions = model.predict(images)

    predicted_labels = np.argmax(predictions, axis=1)

    y_true.extend(labels.numpy())

    y_pred.extend(predicted_labels)

# =========================================================
# 14. METRICS
# =========================================================

print("\nClassification Report:\n")

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names
))

print("\nConfusion Matrix:\n")

print(confusion_matrix(y_true, y_pred))

precision = precision_score(
    y_true,
    y_pred,
    average='weighted'
)

recall = recall_score(
    y_true,
    y_pred,
    average='weighted'
)

f1 = f1_score(
    y_true,
    y_pred,
    average='weighted'
)

print(f"\nPrecision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

loss, accuracy = model.evaluate(test_ds)

print(f"\nTest Accuracy: {accuracy:.4f}")

# =========================================================
# 15. SAVE MODEL
# =========================================================

model.save("/content/drive/MyDrive/MobileNetV3Large_Final.keras")

print("\nModel Saved Successfully")

# =========================================================
# 16. CONVERT TO TFLITE
# =========================================================

converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

with open("/content/drive/MyDrive/currency_model.tflite", "wb") as f:
    f.write(tflite_model)

print("\nTFLite Model Saved Successfully")

# =========================================================
# 17. DOWNLOAD MODEL
# =========================================================

files.download("currency_model.tflite")

# =========================================================
# 18. PLOT ACCURACY GRAPH
# =========================================================

plt.figure(figsize=(10,5))

plt.plot(history_head.history['accuracy'])
plt.plot(history_head.history['val_accuracy'])

plt.title('Training Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')

plt.legend(['Train', 'Validation'])

plt.show()

# =========================================================
# 19. IMAGE UPLOADER FOR TESTING
# =========================================================

print("\n===================================")
print("UPLOAD IMAGE FOR PREDICTION")
print("===================================")

uploaded = files.upload()

# =========================================================
# 20. PREDICTION FUNCTION
# =========================================================

for filename in uploaded.keys():

    image = Image.open(filename).convert("RGB")

    image = image.resize((224, 224))

    image_array = np.array(image).astype("float32")

    image_array = np.expand_dims(image_array, axis=0)

    image_array = tf.keras.applications.imagenet_utils.preprocess_input(
        image_array
    )

    prediction = model.predict(image_array)

    predicted_class = np.argmax(prediction)

    confidence = np.max(prediction) * 100

    print("\n===================================")
    print(f"Prediction: {class_names[predicted_class]}")
    print(f"Confidence: {confidence:.2f}%")
    print("===================================")

    plt.imshow(image)
    plt.axis("off")
    plt.show()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found 4808 files belonging to 2 classes.
Found 601 files belonging to 2 classes.
Found 602 files belonging to 2 classes.

Classes: ['fake', 'real']

TRAINING HEAD
Epoch 1/15
151/151 ━━━━━━━━━━━━━━━━━━━━ 507s 1s/step - accuracy: 0.9133 - loss: 0.2197 - val_accuracy: 0.9351 - val_loss: 0.1562 - learning_rate: 0.0010
Epoch 2/15
151/151 ━━━━━━━━━━━━━━━━━━━━ 8s 54ms/step - accuracy: 0.9659 - loss: 0.0950 - val_accuracy: 0.9700 - val_loss: 0.0793 - learning_rate: 0.0010
Epoch 3/15
151/151 ━━━━━━━━━━━━━━━━━━━━ 8s 55ms/step - accuracy: 0.9750 - loss: 0.0688 - val_accuracy: 0.9784 - val_loss: 0.0588 - learning_rate: 0.0010
Epoch 4/15
151/151 ━━━━━━━━━━━━━━━━━━━━ 8s 52ms/step - accuracy: 0.9796 - loss: 0.0548 - val_accuracy: 0.9767 - val_loss: 0.0531 - learning_rate: 0.0010
Epoch 5/15
151/151 ━━━━━━━━━━━━━━━━━━━━ 10s 48ms/step - accuracy: 0.9859 - loss: 0.0365 - val_ac

ConverterError: Could not translate MLIR to FlatBuffer.<unknown>:0: error: loc(callsite(fused["StridedSlice:", "functional_1_1/strided_slice@__inference_function_63825"] at callsite(fused["StatefulPartitionedCall:", "StatefulPartitionedCall@__inference_signature_wrapper_64920"] at fused["StatefulPartitionedCall:", "StatefulPartitionedCall_1"]))): 'tf.StridedSlice' op is neither a custom op nor a flex op
<unknown>:0: note: loc(callsite(fused["StatefulPartitionedCall:", "StatefulPartitionedCall@__inference_signature_wrapper_64920"] at fused["StatefulPartitionedCall:", "StatefulPartitionedCall_1"])): called from
<unknown>:0: note: loc(fused["StatefulPartitionedCall:", "StatefulPartitionedCall_1"]): called from
<unknown>:0: note: loc(callsite(fused["StridedSlice:", "functional_1_1/strided_slice@__inference_function_63825"] at callsite(fused["StatefulPartitionedCall:", "StatefulPartitionedCall@__inference_signature_wrapper_64920"] at fused["StatefulPartitionedCall:", "StatefulPartitionedCal

In [3]:
import tensorflow as tf

# LOAD MODEL

model = tf.keras.models.load_model(
    "/content/drive/MyDrive/MobileNetV3Large_Final.keras"
)

print("Model Loaded Successfully")

# TFLITE CONVERTER

converter = tf.lite.TFLiteConverter.from_keras_model(model)

# IMPORTANT FIXES

converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]

converter._experimental_lower_tensor_list_ops = False

# CONVERT

tflite_model = converter.convert()

# SAVE MODEL

with open(
    "/content/drive/MyDrive/currency_model.tflite",
    "wb"
) as f:
    f.write(tflite_model)

print("TFLite Model Saved Successfully")

Model Loaded Successfully
Saved artifact at '/tmp/tmpvfdy76d0'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  140681395222480: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float16, name=None)
  140681407116880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140681407115728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140681407115920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140681407116304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140681407115152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140681407114768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140681407114576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140681407115536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140681407114960: TensorSpec(shape=(), dtype=tf.r